# Indonesian Data Job Market Intelligence
## Pipeline Ekstraksi Data Lowongan Kerja: Glints Indonesia (Target Skala 250–500 Loker Bersih)

Notebook ini mengimplementasikan pipeline ekstraksi data lowongan kerja bidang data dari portal **Glints Indonesia** pada skala target **250–500 lowongan unik berkualitas** sesuai panduan proyek (*Panduan Ekstraksi Data Lowongan Kerja*).

### Peningkatan Skala Ekstraksi, Autentikasi & Paginasi:
1. **Ekspansi Taksonomi Kata Kunci (12 Peran):** Mengembangkan kata kunci pencarian ke 12 peran data (5 peran inti, 5 peran spesialis fungsional, dan 2 peran berbasis alat data).
2. **Paginasi Multi-Halaman Eksplisit (*Deep Multi-Page Pagination*):** Menambahkan loop penjelajahan parameter `&page=1`, `&page=2`, `&page=3`, hingga `&page=4` pada URL pencarian Glints guna mengambil lowongan lebih dalam tanpa tertahan di halaman pertama.
3. **Autentikasi & Penanganan Login Wall:** Menyediakan fungsi login manual/interaktif pada peramban non-headless (`HEADLESS_MODE = False`) dengan penyimpanan cookie sesi (`glints_session_cookies.pkl`), sehingga eksplorasi halaman dalam tidak terblokir modal login wall.
4. **Penanganan Duplikasi Ketat di Tahap 1 (*Stage-1 Canonical Deduplication*):**
   - Mengekstrak UUID kanonikal 36-karakter dari slug URL dan membersihkan parameter query tracking (`utm_referrer`, `traceInfo`).
   - Mencegah postingan ganda akibat pekerjaan yang muncul di berbagai kata kunci pencarian agar tidak memasuki antrean scraping detail.
5. **Surgical Detail Extraction:** Mengambil detail lowongan secara presisi dengan mengekstrak judul, perusahaan, lokasi breadcrumb ([Provinsi, Kota]), tipe pekerjaan, gaji, tanggal posting relatif, serta badge sistem `education_raw` dan `experience_raw`.
6. **Checkpointing & Polite Crawling:** Menyimpan progres ke file CSV setiap 5 lowongan dan menerapkan jeda acak manusia (2.0 – 4.0 detik).
7. **Audit Ekstraksi:** Memvalidasi kelengkapan data dan memastikan 0% duplikasi pada data mentah final.

### Sel 1: Inisialisasi Dependensi & Pustaka

In [1]:
import os
import sys
import re
import json
import time
import pickle
import random
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus, urljoin, urlparse

# Konsistensi encoding terminal Windows
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Dependensi Glints MVP berhasil dimuat.")

[2026-10-31 22:04:25] Dependensi Glints MVP berhasil dimuat.


### Sel 2: Konfigurasi Parameter MVP & Jalur Direktori

In [2]:
# Konfigurasi Direktori Proyek
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "extract_data" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# File Checkpoint MVP & Sesi Cookie
CHECKPOINT_LISTINGS_PATH = RAW_DIR / "glints_checkpoint_listings.csv"
CHECKPOINT_DETAILS_PATH = RAW_DIR / "glints_checkpoint_details.csv"
COOKIES_PATH = RAW_DIR / "glints_session_cookies.pkl"

TARGET_KEYWORDS = [
    # Peran Utama (Core Data Roles)
    "Data Analyst",
    "Data Engineer",
    "Data Scientist",
    "Business Intelligence ",
    "Machine Learning Engineer",
    "AI Engineer"
]

COUNTRY_CODE = "ID"
MAX_CARDS_PER_KEYWORD = 100  # Kuota kartu per keyword
MAX_PAGES_PER_KEYWORD = 4   # Batas kedalaman halaman eksplorasi per kata kunci
HEADLESS_MODE = False       # Set False untuk login visual & pemantauan interaktif
POLITE_DELAY_MIN = 2.0
POLITE_DELAY_MAX = 3.8

print(f"Project Root          : {PROJECT_ROOT}")
print(f"Raw Data Dir          : {RAW_DIR}")
print(f"Target Keywords       : {len(TARGET_KEYWORDS)} peran ({TARGET_KEYWORDS})")
print(f"Quota per Keyword     : {MAX_CARDS_PER_KEYWORD} cards (Maks: {MAX_PAGES_PER_KEYWORD} halaman)")
print(f"Potensi Pemindaian    : {len(TARGET_KEYWORDS) * MAX_CARDS_PER_KEYWORD} kartu")
print(f"Headless Mode         : {HEADLESS_MODE}")
print(f"Session Cookies       : {COOKIES_PATH}")

Project Root          : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia
Raw Data Dir          : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw
Target Keywords       : 6 peran (['Data Analyst', 'Data Engineer', 'Data Scientist', 'Business Intelligence ', 'Machine Learning Engineer', 'AI Engineer'])
Quota per Keyword     : 100 cards (Maks: 4 halaman)
Potensi Pemindaian    : 600 kartu
Headless Mode         : False
Session Cookies       : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw\glints_session_cookies.pkl


### Sel 3: Fungsi Helper Utilitas & Deduplikasi Kanonikal

In [3]:
def text_or_none(element, separator=" ", strip=True):
    # Mengekstrak teks dari elemen BeautifulSoup dengan penanganan None aman
    if element is None:
        return None
    val = element.get_text(separator, strip=strip)
    return val if val else None


def clean_glints_url_and_id(raw_href: str) -> tuple[str, str]:
    # Membersihkan URL Glints dari tracking query (?utm_referrer=..., traceInfo=...)
    # dan mengekstrak UUID unik 36-karakter sebagai source_job_id kanonikal.
    if not raw_href:
        return "", ""
    clean_href = raw_href.split("?")[0].strip()
    full_url = urljoin("https://glints.com", clean_href)
    
    # Ekstrak UUID 36 karakter dari akhir path
    m = re.search(r"([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})", clean_href)
    if m:
        job_id = m.group(1)
    else:
        parts = clean_href.strip("/").split("/")
        job_id = parts[-1] if parts else clean_href
        
    return full_url, job_id


def save_checkpoint(df: pd.DataFrame, path: Path, deduplicate_col: str = "source_job_id"):
    # Menyimpan progres checkpoint ke CSV dengan deduplikasi otomatis.
    if df.empty:
        return
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, df], ignore_index=True)
        if deduplicate_col in combined.columns:
            combined = combined.drop_duplicates(subset=[deduplicate_col], keep="last")
        combined.to_csv(path, index=False, encoding="utf-8")
    else:
        df.to_csv(path, index=False, encoding="utf-8")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checkpoint diperbarui: {path.name} ({len(pd.read_csv(path))} baris tersimpan)")

### Sel 4: Inisialisasi WebDriver Selenium

In [4]:
def create_driver(headless: bool = True) -> webdriver.Chrome:
    # Menginisialisasi WebDriver Chrome dengan konfigurasi stabil.
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(35)
    return driver

driver = create_driver(headless=HEADLESS_MODE)
print(f"[{datetime.now().strftime('%H:%M:%S')}] Chrome WebDriver aktif (Headless={HEADLESS_MODE}).")

[22:04:26] Chrome WebDriver aktif (Headless=False).


### Sel 5: Autentikasi Pengguna & Penanganan Login Wall Glints
Glints membatasi eksplorasi daftar lowongan yang lebih dalam jika pengguna belum login (*login wall modal*). 
Sel ini membuka peramban Chrome untuk memungkinkan login secara manual (Google, LinkedIn, atau Email), lalu menyimpan cookie sesi ke `glints_session_cookies.pkl` agar sesi dapat dipulihkan secara otomatis pada eksekusi berikutnya.

In [5]:
def handle_glints_login(driver: webdriver.Chrome, cookies_path: Path = COOKIES_PATH, force_interactive: bool = False) -> bool:
    # Mengelola sesi login Glints untuk menghindari pembatasan akses (login wall modal).
    # Mendukung restorasi cookie sesi otomatis atau panduan login interaktif di browser.
    glints_home = "https://glints.com/id"
    login_url = "https://glints.com/id/login"
    
    # 1. Coba pulihkan sesi dari file cookies jika sudah ada
    if cookies_path.exists() and not force_interactive:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Memuat cookie sesi tersimpan dari: {cookies_path.name}")
        try:
            driver.get(glints_home)
            time.sleep(2.0)
            with open(cookies_path, "rb") as f:
                cookies = pickle.load(f)
            for cookie in cookies:
                c = cookie.copy()
                if "expiry" in c and isinstance(c["expiry"], float):
                    c["expiry"] = int(c["expiry"])
                try:
                    driver.add_cookie(c)
                except Exception:
                    pass
            driver.refresh()
            time.sleep(2.5)
            
            # Periksa apakah sudah dalam kondisi terautentikasi
            soup = BeautifulSoup(driver.page_source, "html.parser")
            login_btns = soup.find_all(lambda tag: tag.name in ["a", "button"] and "masuk" in tag.get_text().lower())
            if not login_btns or len(driver.get_cookies()) > 5:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Sesi login berhasil dipulihkan dari cookie.")
                return True
        except Exception as e:
            print(f"Peringatan: Gagal memulihkan cookie sesi ({e}). Melanjutkan ke login manual.")
            
    # 2. Jika belum login atau cookie kadaluarsa, arahkan ke login interaktif
    print("\n" + "=" * 65)
    print("  TAHAP AUTENTIKASI / LOGIN GLINTS INDONESIA")
    print("=" * 65)
    print("Glints membatasi penjelajahan lowongan jika pengguna belum login.")
    print("Browser Chrome diarahkan ke halaman login Glints:")
    print("  -> URL: https://glints.com/id/login")
    print("\nLangkah yang perlu Anda lakukan:")
    print("1. Pada jendela browser Chrome yang terbuka, silakan login ke akun Glints Anda")
    print("   (bisa via Google SSO, LinkedIn, atau Email/Sandi).")
    print("2. Selesaikan verifikasi keamanan/CAPTCHA jika diminta oleh sistem Glints.")
    print("3. Setelah berhasil masuk dan halaman mengarah ke beranda / dashboard Glints...")
    print("4. Tekan tombol [ENTER] pada prompt input di bawah ini untuk melanjutkan ekstraksi.")
    print("=" * 65 + "\n")
    
    driver.get(login_url)
    
    try:
        input("Tekan [ENTER] di sini SETELAH Anda berhasil login di peramban Chrome...")
    except EOFError:
        print("Lingkungan non-interaktif terdeteksi, menunggu 25 detik untuk penyelesaian login...")
        time.sleep(25)
        
    # 3. Simpan cookies sesi baru setelah pengguna berhasil login
    try:
        cookies = driver.get_cookies()
        with open(cookies_path, "wb") as f:
            pickle.dump(cookies, f)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Sesi login berhasil disimpan ke {cookies_path.name} ({len(cookies)} cookies).")
        return True
    except Exception as e:
        print(f"Peringatan: Gagal menyimpan cookie sesi: {e}")
        return False


# Eksekusi pengecekan / penanganan login
if not HEADLESS_MODE:
    is_logged_in = handle_glints_login(driver, COOKIES_PATH)
else:
    print("Mode Headless aktif. Jika file cookie tersedia, sesi akan dimuat otomatis.")
    if COOKIES_PATH.exists():
        handle_glints_login(driver, COOKIES_PATH)

[22:04:26] Memuat cookie sesi tersimpan dari: glints_session_cookies.pkl
[22:04:34] Sesi login berhasil dipulihkan dari cookie.


### Sel 6: Tahap 1 – Parser Daftar Kartu Lowongan (*Listing Cards*) dengan Deduplikasi & Anti-Modal

In [6]:
def dismiss_glints_modal(driver: webdriver.Chrome):
    # Menutup modal atau pop-up registrasi/iklan yang berpotensi menghalangi halaman pencarian.
    try:
        close_buttons = driver.find_elements(By.XPATH, "//button[contains(@aria-label, 'close') or contains(@aria-label, 'Close') or contains(@class, 'Close')]")
        for btn in close_buttons:
            if btn.is_displayed():
                btn.click()
                time.sleep(0.5)
                return
    except Exception:
        pass
        
    try:
        driver.execute_script('''
            var modals = document.querySelectorAll('div[class*="Modal"], div[class*="Backdrop"], div[class*="Overlay"]');
            modals.forEach(function(m) {
                var txt = (m.innerText || '').toLowerCase();
                if (txt.includes('masuk') || txt.includes('login') || txt.includes('daftar') || txt.includes('sign up')) {
                    m.remove();
                }
            });
            document.body.style.overflow = 'auto';
        ''')
    except Exception:
        pass


def parse_listing_card(a_tag, keyword: str) -> dict:
    # Mengekstrak informasi dasar dari satu elemen kartu lowongan Glints.
    raw_href = a_tag.get("href", "")
    full_url, source_job_id = clean_glints_url_and_id(raw_href)
    title_preview = text_or_none(a_tag.find("h3")) or text_or_none(a_tag)
    
    return {
        "source": "glints",
        "source_job_id": source_job_id,
        "job_url": full_url,
        "title_preview": title_preview,
        "search_keyword": keyword,
        "extracted_at": datetime.now().isoformat()
    }


def extract_glints_listings(driver: webdriver.Chrome, keyword: str, max_cards: int = 70, max_pages: int = 4) -> list[dict]:
    # Mengambil daftar URL kartu lowongan unik untuk kata kunci tertentu dengan paginasi multi-halaman eksplisit.
    encoded_kw = quote_plus(keyword)
    cards = []
    seen_in_kw = set()
    
    print(f"\n--- Mengambil listing Glints: '{keyword}' (Target: {max_cards} kartu, Maks: {max_pages} halaman) ---")
    
    for page in range(1, max_pages + 1):
        search_url = f"https://glints.com/id/opportunities/jobs/explore?keyword={encoded_kw}&country={COUNTRY_CODE}&page={page}"
        print(f"Membuka halaman {page}: {search_url}")
        
        try:
            driver.get(search_url)
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(random.uniform(POLITE_DELAY_MIN, POLITE_DELAY_MAX))
        except Exception as e:
            print(f"Peringatan timeout halaman listing '{keyword}' hal {page}: {e}")
            break
            
        # Scroll bertahap untuk memicu pemuatan kartu lowongan & bersihkan modal jika muncul
        for _ in range(4):
            dismiss_glints_modal(driver)
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(1.0)
            
        dismiss_glints_modal(driver)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        link_elements = soup.find_all("a", href=re.compile(r"/opportunities/jobs/[a-zA-Z0-9-]+"))
        
        page_new = 0
        for a_tag in link_elements:
            href = a_tag.get("href", "")
            clean_url, job_id = clean_glints_url_and_id(href)
            if not job_id or job_id in seen_in_kw:
                continue
            seen_in_kw.add(job_id)
            card_data = parse_listing_card(a_tag, keyword)
            cards.append(card_data)
            page_new += 1
            if len(cards) >= max_cards:
                break
                
        print(f"Halaman {page}: Mendapatkan {page_new} kartu baru (Total: {len(cards)}/{max_cards})")
        if page_new == 0 or len(cards) >= max_cards:
            break
            
        time.sleep(random.uniform(1.5, 2.5))
        
    print(f"Kata kunci '{keyword}': Selesai mengumpulkan {len(cards)} kartu lowongan.")
    return cards

### Sel 7: Tahap 2 – Parser Detail Lowongan (*Surgical Detail Parser*)

In [7]:
def parse_job_detail(html: str, job_url: str, source_job_id: str = None) -> dict:
    # Mengekstrak detail lowongan kerja secara presisi (surgical parsing) dari Glints tanpa memboroskan data.
    soup = BeautifulSoup(html, "html.parser")
    
    title_raw = None
    company_raw = None
    location_raw = None
    salary_raw = None
    job_type_raw = None
    posted_date_raw = None
    education_raw = None
    experience_raw = None
    description_raw = None
    
    # 1. Judul Pekerjaan
    h1_el = soup.find("h1")
    if h1_el:
        title_raw = text_or_none(h1_el)
        
    # 2. Lokasi Presisi dari Breadcrumb (Dibatasi 2 tingkat teratas untuk mencegah duplikasi)
    breadcrumb_links = soup.find_all("a", href=re.compile(r"/job-location/indonesia/"))
    if breadcrumb_links:
        loc_parts = []
        seen_loc = set()
        for a in breadcrumb_links:
            txt = text_or_none(a)
            if txt and txt.lower() not in seen_loc:
                seen_loc.add(txt.lower())
                loc_parts.append(txt)
                if len(loc_parts) >= 2:
                    break
        if loc_parts:
            location_raw = ", ".join(loc_parts)
            
    # 3. Identifikasi Nama Perusahaan
    for a_tag in soup.find_all("a", href=re.compile(r"/companies/")):
        txt = text_or_none(a_tag)
        if txt and len(txt) > 1 and "glints" not in txt.lower():
            company_raw = txt
            break
            
    # 4. Gaji Presisi (Surgical Salary Extraction)
    # Diambil langsung dari span BasicSalary atau kontainer spesifik SalaryJobOverview
    salary_el = soup.find("span", class_=re.compile(r"BasicSalary", re.I)) or soup.find("div", class_=re.compile(r"SalaryJobOverview", re.I))
    if salary_el:
        sal_text = text_or_none(salary_el)
        if sal_text and len(sal_text) < 100:
            salary_raw = sal_text
            
    # 5. Ekstraksi Badge TopFold (Pendidikan, Pengalaman, Tipe Pekerjaan) dari JobOverViewInfo
    overview_items = soup.find_all("div", class_=re.compile(r"JobOverViewInfo", re.I))
    for item in overview_items:
        txt = text_or_none(item)
        if not txt or len(txt) > 120:  # Batasi agar tidak serakah mengambil kontainer besar/seluruh halaman
            continue
            
        # A. Pendidikan (GraduationHatIcon atau teks kualifikasi pendidikan terstruktur)
        has_hat = item.find(lambda t: t.name == "svg" and any("graduationhat" in str(c).lower() for c in t.get("class", [])))
        if not education_raw and (has_hat or any(edu in txt for edu in ["Minimal Sarjana", "Sarjana (S1)", "Minimal Diploma", "Diploma (D3)", "Minimal SMA", "SMA/SMK", "Magister (S2)", "Doktor (S3)", "Gelar Sarjana", "Diploma", "Sarjana"])):
            span_el = item.find("span")
            education_raw = text_or_none(span_el) if span_el else txt
            continue
            
        # B. Pengalaman (Pola tahun pengalaman atau fresh graduate)
        if not experience_raw and re.search(r"(\d+\s*-\s*\d+\s*tahun|kurang dari \d+|lebih dari \d+|tahun pengalaman|fresh graduate)", txt, re.I):
            experience_raw = txt
            continue
            
        # C. Tipe Pekerjaan
        if not job_type_raw and txt in ["Penuh Waktu", "Full-time", "Paruh Waktu", "Part-time", "Kontrak", "Contract", "Magang", "Internship"]:
            job_type_raw = txt
            continue
            
        # D. Fallback Gaji jika belum tertangkap dari BasicSalary
        if not salary_raw and re.search(r"\b(idr|rp\.?|usd|\$)\b", txt, re.IGNORECASE) and re.search(r"\d", txt):
            if any(term in txt.lower() for term in ["per bulan", "bulan", "/ bulan"]):
                salary_raw = txt
                continue
                
    # 6. Tanggal Posting Relatif (Dibatasi panjang < 80 karakter agar tidak serakah)
    posted_el = soup.find(lambda t: t.name in ["span", "p", "div"] and len(t.get_text(strip=True)) < 80 and re.search(r"(?:diperbarui|diposting)\s+\d+.*(?:lalu|ago)", t.get_text(strip=True), re.I))
    if posted_el:
        posted_date_raw = text_or_none(posted_el)
            
    # 7. Deskripsi Pekerjaan Lengkap
    desc_header = soup.find(lambda t: t.name in ["h2", "h3"] and "deskripsi pekerjaan" in t.get_text().lower())
    if desc_header:
        curr = desc_header
        while curr and curr.parent and curr.parent.name not in ["body", "html", "[document]"] and len(curr.get_text(strip=True)) < 200:
            curr = curr.parent
        if curr:
            description_raw = curr.get_text(separator="\n", strip=True)
            
    if not description_raw:
        main_tag = soup.find("main")
        if main_tag:
            description_raw = main_tag.get_text(separator="\n", strip=True)
            
    parse_status = "OK" if description_raw else "EMPTY_DESCRIPTION"
    parse_error = None if description_raw else "Deskripsi pekerjaan tidak ditemukan"
    
    return {
        "source": "glints",
        "source_job_id": source_job_id,
        "job_url": job_url,
        "title_raw": title_raw,
        "company_raw": company_raw,
        "location_raw": location_raw,
        "salary_raw": salary_raw,
        "job_type_raw": job_type_raw,
        "education_raw": education_raw,
        "experience_raw": experience_raw,
        "posted_date_raw": posted_date_raw,
        "description_raw": description_raw,
        "collected_at": datetime.now().isoformat(),
        "parse_status": parse_status,
        "parse_error": parse_error
    }

### Sel 8: Eksekusi Pipeline Ekstraksi MVP dengan Deduplikasi Ketat

In [8]:
# ==========================================
# 1. Kumpulkan daftar lowongan unik (Tahap 1)
# ==========================================
all_listings = []
seen_job_ids = set()
seen_urls = set()
duplicates_skipped = 0

# Muat checkpoint listing jika sudah ada
if CHECKPOINT_LISTINGS_PATH.exists():
    df_existing_listings = pd.read_csv(CHECKPOINT_LISTINGS_PATH)
    if 'source_job_id' in df_existing_listings:
        seen_job_ids.update(df_existing_listings['source_job_id'].dropna().astype(str).tolist())
    seen_urls.update(df_existing_listings['job_url'].dropna().tolist())
    all_listings.extend(df_existing_listings.to_dict('records'))
    print(f"Melanjutkan dari checkpoint listing: {len(seen_job_ids)} lowongan tersimpan.")

for kw in TARGET_KEYWORDS:
    cards = extract_glints_listings(driver, kw, max_cards=MAX_CARDS_PER_KEYWORD, max_pages=MAX_PAGES_PER_KEYWORD)
    new_cards = []
    for c in cards:
        jid = str(c.get('source_job_id', ''))
        jurl = c.get('job_url', '')
        if jid in seen_job_ids or jurl in seen_urls:
            duplicates_skipped += 1
            continue
        seen_job_ids.add(jid)
        seen_urls.add(jurl)
        all_listings.append(c)
        new_cards.append(c)
        
    print(f"Kata kunci '{kw}': +{len(new_cards)} lowongan baru ditambahkan.")
    if new_cards:
        save_checkpoint(pd.DataFrame(all_listings), CHECKPOINT_LISTINGS_PATH)
    time.sleep(random.uniform(POLITE_DELAY_MIN, POLITE_DELAY_MAX))

print(f"\nTotal listing unik terdeduplikasi: {len(all_listings)} lowongan (Duplikasi lintas keyword dicegah: {duplicates_skipped}).")

# ==========================================
# 2. Ambil detail setiap lowongan (Tahap 2)
# ==========================================
details_records = []
processed_ids = set()

if CHECKPOINT_DETAILS_PATH.exists():
    df_existing_details = pd.read_csv(CHECKPOINT_DETAILS_PATH)
    if 'source_job_id' in df_existing_details:
        processed_ids.update(df_existing_details['source_job_id'].dropna().astype(str).tolist())
    details_records.extend(df_existing_details.to_dict('records'))
    print(f"Melanjutkan dari checkpoint detail: {len(processed_ids)} detail sudah diproses.")

# Filter antrean yang belum diproses
pending_listings = [c for c in all_listings if str(c.get('source_job_id', '')) not in processed_ids]
total_queue = len(pending_listings)
print(f"Antrean detail yang akan diambil: {total_queue} lowongan.\n")

for idx, card in enumerate(pending_listings, start=1):
    url = card['job_url']
    job_id = card.get('source_job_id')
    print(f"[{idx}/{total_queue}] Mengambil: {url}")
    
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        time.sleep(random.uniform(POLITE_DELAY_MIN, POLITE_DELAY_MAX))
        
        detail_data = parse_job_detail(driver.page_source, url, source_job_id=job_id)
        if not detail_data.get('title_raw') and card.get('title_preview'):
            detail_data['title_raw'] = card['title_preview']
        detail_data['search_keyword'] = card.get('search_keyword')
        
        details_records.append(detail_data)
        processed_ids.add(str(job_id))
    except Exception as e:
        print(f"  [ERROR] Gagal memproses {url}: {e}")
        details_records.append({
            "source": "glints",
            "source_job_id": job_id,
            "job_url": url,
            "title_raw": card.get('title_preview'),
            "company_raw": None,
            "location_raw": None,
            "salary_raw": None,
            "job_type_raw": None,
            "education_raw": None,
            "experience_raw": None,
            "posted_date_raw": None,
            "description_raw": None,
            "search_keyword": card.get('search_keyword'),
            "collected_at": datetime.now().isoformat(),
            "parse_status": "EXCEPTION",
            "parse_error": str(e)
        })
        
    if idx % 5 == 0 or idx == total_queue:
        save_checkpoint(pd.DataFrame(details_records), CHECKPOINT_DETAILS_PATH)
        
    time.sleep(random.uniform(1.0, 2.0))

df_raw_final = pd.DataFrame(details_records)
print(f"\nEkstraksi Glints MVP Selesai. Total lowongan mentah terdeduplikasi: {len(df_raw_final)}.")


--- Mengambil listing Glints: 'Data Analyst' (Target: 100 kartu, Maks: 4 halaman) ---
Membuka halaman 1: https://glints.com/id/opportunities/jobs/explore?keyword=Data+Analyst&country=ID&page=1
Halaman 1: Mendapatkan 61 kartu baru (Total: 61/100)
Membuka halaman 2: https://glints.com/id/opportunities/jobs/explore?keyword=Data+Analyst&country=ID&page=2
Halaman 2: Mendapatkan 0 kartu baru (Total: 61/100)
Kata kunci 'Data Analyst': Selesai mengumpulkan 61 kartu lowongan.
Kata kunci 'Data Analyst': +61 lowongan baru ditambahkan.
[22:04:54] Checkpoint diperbarui: glints_checkpoint_listings.csv (61 baris tersimpan)

--- Mengambil listing Glints: 'Data Engineer' (Target: 100 kartu, Maks: 4 halaman) ---
Membuka halaman 1: https://glints.com/id/opportunities/jobs/explore?keyword=Data+Engineer&country=ID&page=1
Halaman 1: Mendapatkan 61 kartu baru (Total: 61/100)
Membuka halaman 2: https://glints.com/id/opportunities/jobs/explore?keyword=Data+Engineer&country=ID&page=2
Halaman 2: Mendapatkan 2 k

### Sel 9: Audit Kualitas Ekstraksi Data Mentah (`audit_extraction`)

In [9]:
def audit_extraction(df: pd.DataFrame) -> pd.Series:
    # Menghasilkan ringkasan audit kualitas data mentah (Bab 15 Panduan)
    total = len(df)
    summary = {
        "total_rows": total,
        "missing_title": df["title_raw"].isna().sum() if "title_raw" in df else 0,
        "missing_company": df["company_raw"].isna().sum() if "company_raw" in df else 0,
        "missing_location": df["location_raw"].isna().sum() if "location_raw" in df else 0,
        "missing_salary": df["salary_raw"].isna().sum() if "salary_raw" in df else 0,
        "missing_description": df["description_raw"].isna().sum() if "description_raw" in df else 0,
        "missing_posted_date": df["posted_date_raw"].isna().sum() if "posted_date_raw" in df else 0,
        "parse_success_count": (df["parse_status"] == "OK").sum() if "parse_status" in df else 0,
        "duplicate_urls": df["job_url"].duplicated().sum() if "job_url" in df else 0,
        "duplicate_job_ids": df["source_job_id"].dropna().duplicated().sum() if "source_job_id" in df else 0
    }
    return pd.Series(summary)

print("=== RINGKASAN AUDIT KUALITAS EKSTRAKSI GLINTS MVP ===")
audit_result = audit_extraction(df_raw_final)
display(audit_result.to_frame(name="Nilai"))

print("\nDistribusi Kata Kunci:")
display(df_raw_final["search_keyword"].value_counts().to_frame(name="Jumlah"))

print("\nSampel 5 Baris Data Mentah:")
display(df_raw_final[["source_job_id", "title_raw", "company_raw", "location_raw", "salary_raw", "parse_status"]].head(5))

=== RINGKASAN AUDIT KUALITAS EKSTRAKSI GLINTS MVP ===


,Nilai
total_rows,319
missing_title,0
missing_company,0
missing_location,0
missing_salary,1
missing_description,0
missing_posted_date,319
parse_success_count,319
duplicate_urls,0
duplicate_job_ids,0



Distribusi Kata Kunci:


,Jumlah
search_keyword,
Machine Learning Engineer,92
Data Engineer,64
Data Analyst,61
Business Intelligence,52
AI Engineer,40
Data Scientist,10



Sampel 5 Baris Data Mentah:


,source_job_id,title_raw,company_raw,location_raw,salary_raw,parse_status
0,recommended,Pekerjaan yang direkomendasikan untuk kamu,SoftwareSeni,"Yogyakarta, DI Yogyakarta, Yogyakarta, DI Yogy...",None,OK
1,a0726783-7124-4916-8797-52a6bee58e07,Data Analyst,Nusantara Sakti Group,"Jawa Tengah, Semarang, Surabaya, Jawa Timur, S...",Rp 7.000.000 - 10.000.000 / Bulan,OK
2,07401ee8-bc95-483c-a0ba-ba1779d88770,Data Analyst,PT Sigma Digital Nusantara,"DI Yogyakarta, Yogyakarta, Kab. Sukoharjo, Jaw...",Perusahaan tidak menampilkan gaji,OK
3,f531ab9c-2168-43ee-bc0e-2c6175c326ba,Data Analyst Intern,"PT Tunas Ridean, Tbk","DKI Jakarta, Jakarta Selatan, Jakarta Selatan,...",Perusahaan tidak menampilkan gaji,OK
4,42869c9f-8db6-47eb-aea4-370996ecff03,Data Analyst,MSBU,"DKI Jakarta, Jakarta Pusat, Jakarta Pusat, DKI...",Perusahaan tidak menampilkan gaji,OK


### Sel 10: Ekspor Dataset Mentah Final & Penutupan WebDriver

In [10]:
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
final_raw_csv = RAW_DIR / f"glints_raw_{timestamp_str}.csv"
final_raw_json = RAW_DIR / f"glints_raw_{timestamp_str}.json"

df_raw_final.to_csv(final_raw_csv, index=False, encoding="utf-8")
df_raw_final.to_json(final_raw_json, orient="records", indent=2, force_ascii=False)

print(f"Data mentah final Glints berhasil diekspor:")
print(f"  - CSV  : {final_raw_csv.resolve()} ({final_raw_csv.stat().st_size / 1024:.1f} KB)")
print(f"  - JSON : {final_raw_json.resolve()} ({final_raw_json.stat().st_size / 1024:.1f} KB)")

try:
    driver.quit()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] WebDriver berhasil ditutup.")
except Exception as e:
    print(f"Catatan penutupan driver: {e}")

Data mentah final Glints berhasil diekspor:
  - CSV  : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw\glints_raw_20261031_223844.csv (669.7 KB)
  - JSON : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw\glints_raw_20261031_223844.json (790.0 KB)
[22:38:47] WebDriver berhasil ditutup.
